# Práctica: Transfer Learning progresivo en Computer Vision

**Módulo:** Computer Vision  
**Tema:** Deep Learning aplicado a imagen  
**Objetivo:** aprender a usar modelos preentrenados y comparar distintas estrategias de congelación y fine tuning.

## Qué vas a practicar

- Entrenar una CNN sencilla desde cero como baseline.
- Usar modelos preentrenados como extractores de características.
- Congelar y descongelar capas.
- Comparar **VGG16**, **ResNet50**, **MobileNetV2** y **EfficientNetB0**.
- Entender para qué sirven **U-Net**, **YOLO**, **R-CNN**, **ViT** y **DETR**.
- Analizar accuracy, loss, número de parámetros, matriz de confusión y coste de entrenamiento.

## Dataset

Usaremos **TensorFlow Flowers**, un dataset de clasificación de flores con 5 clases:

- daisy
- dandelion
- roses
- sunflowers
- tulips

> Recomendado en Google Colab: activar GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.

# 1. Instalación e importación de librerías

In [ ]:
# Si hace falta, descomenta esta línea:
# !pip install tensorflow-datasets

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# 2. Introducción teórica

## Transfer Learning

El **transfer learning** consiste en tomar un modelo entrenado previamente en un gran conjunto de datos y adaptarlo a una nueva tarea.

En Computer Vision suele funcionar muy bien porque las primeras capas de una CNN aprenden patrones generales:

- bordes,
- texturas,
- formas simples,
- patrones locales.

Las capas finales suelen aprender patrones más específicos de la tarea original.

## Estrategias habituales

1. **Modelo congelado:** se congela el modelo base y solo se entrena un clasificador nuevo.
2. **Fine tuning parcial:** se descongelan algunas capas finales.
3. **Fine tuning amplio:** se descongelan más capas, normalmente con learning rate bajo.
4. **Entrenamiento desde cero:** se entrena todo el modelo sin pesos preentrenados.

# 3. Mapa de arquitecturas

| Modelo | Tipo de tarea principal | Idea clave | ¿Lo usaremos? |
|---|---|---|---|
| LeNet | Clasificación | CNN clásica pequeña | No |
| VGG16 | Clasificación | Bloques conv 3x3 | Sí |
| ResNet50 | Clasificación | Conexiones residuales | Sí |
| MobileNetV2 | Clasificación ligera | Modelo eficiente para móviles | Sí |
| EfficientNetB0 | Clasificación eficiente | Buen equilibrio precisión/coste | Sí |
| U-Net | Segmentación | Encoder-decoder con skip connections | Solo teoría |
| YOLO | Detección | Cajas y clases en una pasada | Solo teoría |
| R-CNN / Faster R-CNN | Detección | Propuestas de regiones | Solo teoría |
| ViT | Clasificación | Transformer aplicado a parches | Opcional |
| DETR | Detección | Transformer para detección | Solo teoría |

## Preguntas

1. ¿Qué diferencia hay entre clasificación, detección y segmentación?
2. ¿Por qué U-Net no es la arquitectura natural para este ejercicio?
3. ¿Por qué YOLO no es la arquitectura natural para este ejercicio?
4. ¿Qué modelos parecen más adecuados si buscamos eficiencia?
5. ¿Qué modelos parecen más adecuados si buscamos rendimiento?

# 4. Carga del dataset TensorFlow Flowers

In [ ]:
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "tf_flowers",
    split=["train[:70%]", "train[70%:85%]", "train[85%:]"],
    as_supervised=True,
    with_info=True
)

num_classes = ds_info.features["label"].num_classes
class_names = ds_info.features["label"].names

print("Número de clases:", num_classes)
print("Clases:", class_names)

# 5. Visualización inicial

In [ ]:
plt.figure(figsize=(10, 6))
for i, (image, label) in enumerate(ds_train.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Preguntas

1. ¿Son imágenes homogéneas o tienen fondos variados?
2. ¿Todas las flores aparecen centradas?
3. ¿Hay clases visualmente parecidas?
4. ¿Crees que el dataset será más fácil o más difícil que MNIST?
5. ¿Qué problemas puede tener este dataset para entrenar desde cero?

# 6. Configuración común

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
results = []

def resize_only(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

ds_train_resized = ds_train.map(resize_only, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_resized = ds_val.map(resize_only, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_resized = ds_test.map(resize_only, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# 7. Funciones auxiliares

In [ ]:
def plot_history(history, title="Entrenamiento"):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="val")
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


def count_trainable_params(model):
    return int(np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]))


def count_non_trainable_params(model):
    return int(np.sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights]))


def evaluate_and_store(model, test_ds, model_name, strategy, elapsed_time=None):
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    row = {
        "modelo": model_name,
        "estrategia": strategy,
        "accuracy_test": test_acc,
        "loss_test": test_loss,
        "params_totales": model.count_params(),
        "params_entrenables": count_trainable_params(model),
        "params_no_entrenables": count_non_trainable_params(model),
        "tiempo_entrenamiento_seg": elapsed_time
    }
    results.append(row)
    print(pd.Series(row))
    return row

# Parte A — Baseline: CNN desde cero

Antes de usar transfer learning, entrenaremos una CNN sencilla desde cero. Este modelo servirá como referencia.

# 8. Modelo 1 — CNN desde cero

In [ ]:
def build_simple_cnn():
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

simple_cnn = build_simple_cnn()
simple_cnn.summary()

In [ ]:
start = time.time()
history_simple = simple_cnn.fit(ds_train_resized, validation_data=ds_val_resized, epochs=20, callbacks=[early_stop])
elapsed_simple = time.time() - start
plot_history(history_simple, "CNN desde cero")
evaluate_and_store(simple_cnn, ds_test_resized, "CNN desde cero", "Entrenamiento completo", elapsed_simple)

## Preguntas

1. ¿Cuántos parámetros tiene el modelo?
2. ¿Cuántos parámetros son entrenables?
3. ¿Entrena rápido o lento?
4. ¿Hay sobreentrenamiento?
5. ¿Qué accuracy obtiene?
6. ¿Qué limitación tiene entrenar desde cero con pocos datos?

# Parte — Transfer Learning con VGG16

En esta sección usaremos **VGG16** preentrenada en ImageNet como extractor de características.

Primera estrategia:

- Cargar el modelo sin su clasificador final (`include_top=False`).
- Congelar el modelo base.
- Añadir un nuevo clasificador.
- Entrenar solo las capas nuevas.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16

In [ ]:
def preprocess_for_vgg(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_vgg16(image)
    return image, label

ds_train_vgg = ds_train.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_vgg = ds_val.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_vgg = ds_test.map(preprocess_for_vgg, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
base_vgg = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_vgg.trainable = False

model_vgg_frozen = models.Sequential([
    base_vgg,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model_vgg_frozen.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_vgg_frozen.summary()

In [ ]:
start = time.time()
history_vgg_frozen = model_vgg_frozen.fit(ds_train_vgg, validation_data=ds_val_vgg, epochs=15, callbacks=[early_stop])
elapsed_vgg_frozen = time.time() - start
plot_history(history_vgg_frozen, "VGG16 congelada")
evaluate_and_store(model_vgg_frozen, ds_test_vgg, "VGG16", "Base congelada + clasificador nuevo", elapsed_vgg_frozen)

## Preguntas sobre VGG16 congelada

1. ¿Cuántos parámetros totales tiene el modelo?
2. ¿Cuántos parámetros son entrenables?
3. ¿Mejora respecto a la CNN entrenada desde cero?
4. ¿Qué ventaja aporta usar pesos preentrenados?
5. ¿Qué coste computacional observas?

# Fine tuning parcial con VGG16

Ahora descongelaremos solo el último bloque de VGG16: `block5`.

Usaremos un learning rate bajo.

In [ ]:
base_vgg.trainable = True
for layer in base_vgg.layers:
    if not layer.name.startswith("block5"):
        layer.trainable = False

for layer in base_vgg.layers:
    print(layer.name, layer.trainable)

In [ ]:
model_vgg_finetune = models.Sequential([
    base_vgg,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model_vgg_finetune.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_vgg_finetune.summary()

In [ ]:
start = time.time()
history_vgg_finetune = model_vgg_finetune.fit(ds_train_vgg, validation_data=ds_val_vgg, epochs=10, callbacks=[early_stop])
elapsed_vgg_finetune = time.time() - start
plot_history(history_vgg_finetune, "VGG16 fine tuning parcial")
evaluate_and_store(model_vgg_finetune, ds_test_vgg, "VGG16", "Fine tuning block5", elapsed_vgg_finetune)

## Preguntas sobre fine tuning

1. ¿Por qué usamos un learning rate más bajo?
2. ¿Qué puede ocurrir si descongelamos muchas capas con un learning rate alto?
3. ¿Mejora el modelo al hacer fine tuning?
4. ¿Aumenta el tiempo de entrenamiento?
5. ¿Aumenta el riesgo de sobreentrenamiento?

# Parte — Transfer Learning con ResNet50

En esta sección usaremos **ResNet50** preentrenada en ImageNet como extractor de características.

Primera estrategia:

- Cargar el modelo sin su clasificador final (`include_top=False`).
- Congelar el modelo base.
- Añadir un nuevo clasificador.
- Entrenar solo las capas nuevas.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet

In [ ]:
def preprocess_for_resnet(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_resnet(image)
    return image, label

ds_train_resnet = ds_train.map(preprocess_for_resnet, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_resnet = ds_val.map(preprocess_for_resnet, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_resnet = ds_test.map(preprocess_for_resnet, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
base_resnet = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_resnet.trainable = False

model_resnet_frozen = models.Sequential([
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model_resnet_frozen.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_resnet_frozen.summary()

In [ ]:
start = time.time()
history_resnet_frozen = model_resnet_frozen.fit(ds_train_resnet, validation_data=ds_val_resnet, epochs=15, callbacks=[early_stop])
elapsed_resnet_frozen = time.time() - start
plot_history(history_resnet_frozen, "ResNet50 congelada")
evaluate_and_store(model_resnet_frozen, ds_test_resnet, "ResNet50", "Base congelada + clasificador nuevo", elapsed_resnet_frozen)

## Preguntas sobre ResNet50 congelada

1. ¿Cuántos parámetros totales tiene el modelo?
2. ¿Cuántos parámetros son entrenables?
3. ¿Mejora respecto a la CNN entrenada desde cero?
4. ¿Qué ventaja aporta usar pesos preentrenados?
5. ¿Qué coste computacional observas?

# Fine tuning parcial con ResNet50

Ahora descongelaremos aproximadamente las últimas 30 capas.

In [ ]:
base_resnet.trainable = True
for layer in base_resnet.layers[:-30]:
    layer.trainable = False

for layer in base_resnet.layers[-40:]:
    print(layer.name, layer.trainable)

In [ ]:
model_resnet_finetune = models.Sequential([
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model_resnet_finetune.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_resnet_finetune.summary()

In [ ]:
start = time.time()
history_resnet_finetune = model_resnet_finetune.fit(ds_train_resnet, validation_data=ds_val_resnet, epochs=10, callbacks=[early_stop])
elapsed_resnet_finetune = time.time() - start
plot_history(history_resnet_finetune, "ResNet50 fine tuning parcial")
evaluate_and_store(model_resnet_finetune, ds_test_resnet, "ResNet50", "Fine tuning últimas 30 capas", elapsed_resnet_finetune)

## Preguntas sobre fine tuning

1. ¿Por qué usamos un learning rate más bajo?
2. ¿Qué puede ocurrir si descongelamos muchas capas con un learning rate alto?
3. ¿Mejora el modelo al hacer fine tuning?
4. ¿Aumenta el tiempo de entrenamiento?
5. ¿Aumenta el riesgo de sobreentrenamiento?

# Parte — Transfer Learning con MobileNetV2

En esta sección usaremos **MobileNetV2** preentrenada en ImageNet como extractor de características.

Primera estrategia:

- Cargar el modelo sin su clasificador final (`include_top=False`).
- Congelar el modelo base.
- Añadir un nuevo clasificador.
- Entrenar solo las capas nuevas.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenet

In [ ]:
def preprocess_for_mobile(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_mobilenet(image)
    return image, label

ds_train_mobile = ds_train.map(preprocess_for_mobile, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_mobile = ds_val.map(preprocess_for_mobile, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_mobile = ds_test.map(preprocess_for_mobile, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
base_mobile = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_mobile.trainable = False

model_mobile_frozen = models.Sequential([
    base_mobile,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model_mobile_frozen.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_mobile_frozen.summary()

In [ ]:
start = time.time()
history_mobile_frozen = model_mobile_frozen.fit(ds_train_mobile, validation_data=ds_val_mobile, epochs=15, callbacks=[early_stop])
elapsed_mobile_frozen = time.time() - start
plot_history(history_mobile_frozen, "MobileNetV2 congelada")
evaluate_and_store(model_mobile_frozen, ds_test_mobile, "MobileNetV2", "Base congelada + clasificador nuevo", elapsed_mobile_frozen)

## Preguntas sobre MobileNetV2 congelada

1. ¿Cuántos parámetros totales tiene el modelo?
2. ¿Cuántos parámetros son entrenables?
3. ¿Mejora respecto a la CNN entrenada desde cero?
4. ¿Qué ventaja aporta usar pesos preentrenados?
5. ¿Qué coste computacional observas?

# Parte — Transfer Learning con EfficientNetB0

En esta sección usaremos **EfficientNetB0** preentrenada en ImageNet como extractor de características.

Primera estrategia:

- Cargar el modelo sin su clasificador final (`include_top=False`).
- Congelar el modelo base.
- Añadir un nuevo clasificador.
- Entrenar solo las capas nuevas.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficientnet

In [ ]:
def preprocess_for_eff(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_efficientnet(image)
    return image, label

ds_train_eff = ds_train.map(preprocess_for_eff, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val_eff = ds_val.map(preprocess_for_eff, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test_eff = ds_test.map(preprocess_for_eff, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
base_eff = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_eff.trainable = False

model_eff_frozen = models.Sequential([
    base_eff,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model_eff_frozen.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_eff_frozen.summary()

In [ ]:
start = time.time()
history_eff_frozen = model_eff_frozen.fit(ds_train_eff, validation_data=ds_val_eff, epochs=15, callbacks=[early_stop])
elapsed_eff_frozen = time.time() - start
plot_history(history_eff_frozen, "EfficientNetB0 congelada")
evaluate_and_store(model_eff_frozen, ds_test_eff, "EfficientNetB0", "Base congelada + clasificador nuevo", elapsed_eff_frozen)

## Preguntas sobre EfficientNetB0 congelada

1. ¿Cuántos parámetros totales tiene el modelo?
2. ¿Cuántos parámetros son entrenables?
3. ¿Mejora respecto a la CNN entrenada desde cero?
4. ¿Qué ventaja aporta usar pesos preentrenados?
5. ¿Qué coste computacional observas?

# 19. Comparativa final de modelos

In [ ]:
df_results = pd.DataFrame(results)
df_results

In [ ]:
df_results.sort_values("accuracy_test", ascending=False)

## Preguntas de comparación

1. ¿Qué modelo consigue mejor accuracy de test?
2. ¿Qué modelo tiene menos parámetros?
3. ¿Qué modelo tarda menos en entrenar?
4. ¿Qué modelo ofrece mejor equilibrio entre accuracy y coste?
5. ¿Ha merecido la pena hacer fine tuning parcial?
6. ¿Qué modelo elegirías para producción?
7. ¿Qué modelo elegirías para un móvil?
8. ¿Qué modelo elegirías si solo importa la precisión?

# 20. Matriz de confusión del mejor modelo

Por defecto se usa EfficientNetB0. Puedes cambiar `best_model` y `best_test_ds` por el modelo que mejor resultado te haya dado.

In [ ]:
best_model = model_eff_frozen
best_test_ds = ds_test_eff

y_true = []
y_pred = []

for images, labels in best_test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Matriz de confusión - Mejor modelo")
plt.show()

## Preguntas

1. ¿Qué clases se confunden más?
2. ¿Hay alguna clase especialmente fácil?
3. ¿Hay alguna clase especialmente difícil?
4. ¿La confusión tiene sentido visual?
5. ¿Qué harías para mejorar esas clases?

# 21. Mini-reto: congelación progresiva

Escoge uno de estos modelos:

- VGG16
- ResNet50
- MobileNetV2
- EfficientNetB0

Y prueba tres configuraciones:

| Experimento | Capas congeladas | Capas entrenables |
|---|---|---|
| A | Todas las capas base | Solo clasificador |
| B | Todo menos último bloque | Clasificador + último bloque |
| C | Todo menos últimos 2 bloques | Clasificador + últimos 2 bloques |

Para cada experimento, registra:

- Accuracy de entrenamiento.
- Accuracy de validación.
- Accuracy de test.
- Número de parámetros entrenables.
- Tiempo aproximado de entrenamiento.
- Señales de sobreentrenamiento.

In [ ]:
# Plantilla general para el mini-reto

# base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
# base_model.trainable = True

# Congelar todo menos las últimas N capas
# N = 20
# for layer in base_model.layers[:-N]:
#     layer.trainable = False

# model = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation="relu"),
#     layers.Dropout(0.4),
#     layers.Dense(num_classes, activation="softmax")
# ])

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# model.summary()
# history = model.fit(...)
# model.evaluate(...)

# 22. Solución orientativa

## CNN desde cero

La CNN desde cero probablemente funcionará peor que los modelos preentrenados porque el dataset es pequeño. El modelo tiene que aprender desde cero bordes, texturas, formas y patrones discriminativos.

## VGG16 congelada

VGG16 puede funcionar bien como extractor de características, pero es pesada. Es didáctica porque su arquitectura es muy interpretable.

## VGG16 con fine tuning

Al descongelar el último bloque puede mejorar el resultado, pero aumenta el riesgo de sobreentrenamiento. Conviene usar un learning rate bajo.

## ResNet50

ResNet50 suele generalizar bien gracias a sus conexiones residuales, que facilitan el entrenamiento de redes profundas.

## MobileNetV2

MobileNetV2 suele ser muy eficiente. Es adecuada para despliegue en dispositivos con recursos limitados.

## EfficientNetB0

EfficientNetB0 suele ofrecer un equilibrio muy competitivo entre precisión y coste.

## U-Net

U-Net no es adecuada para esta práctica porque está pensada para segmentación. Su salida es una máscara por píxel, no una etiqueta global de imagen.

## YOLO, R-CNN y DETR

No se usan directamente en esta práctica porque son modelos de detección. Su objetivo es localizar objetos mediante cajas delimitadoras, no clasificar la imagen completa.

# 23. Preguntas finales de reflexión

1. ¿Por qué transfer learning suele funcionar mejor que entrenar desde cero?
2. ¿Qué capas de una CNN suelen aprender características más generales?
3. ¿Qué capas suelen aprender características más específicas?
4. ¿Por qué no conviene descongelar toda la red desde el principio?
5. ¿Qué pasa si el learning rate es demasiado alto durante fine tuning?
6. ¿Qué modelo ha dado mejor accuracy?
7. ¿Qué modelo ha sido más eficiente?
8. ¿Cuál elegirías para producción?
9. ¿Cuál elegirías para un móvil?
10. ¿Cuál elegirías si solo te importa la precisión?
11. ¿Por qué U-Net no se usa directamente para clasificación?
12. ¿Qué diferencia hay entre clasificación, detección y segmentación?
13. ¿Qué diferencia hay entre usar un modelo congelado y hacer fine tuning?
14. ¿Por qué puede empeorar el modelo al descongelar demasiadas capas?
15. ¿Qué papel tiene el tamaño del dataset en la estrategia de transfer learning?

# 24. Rúbrica sugerida

| Criterio | Peso |
|---|---:|
| Carga y visualización del dataset | 10% |
| CNN baseline desde cero | 15% |
| Transfer learning con modelo congelado | 20% |
| Fine tuning parcial | 20% |
| Comparación entre arquitecturas | 20% |
| Matriz de confusión y análisis de errores | 10% |
| Conclusiones finales | 5% |